# Minimal Sage Agent Run

In [1]:
%load_ext autoreload
%autoreload 2

import os
import rootutils
from dotenv import load_dotenv

root = rootutils.setup_root(search_from='..', indicator="pyproject.toml", pythonpath=True, cwd=True)
print(f"Root directory: {root}")
print(f".env file path: {root / '.env'}")
load_dotenv(dotenv_path= root / ".env")

Root directory: /Users/snopoff/Documents/Research/LLMxM2
.env file path: /Users/snopoff/Documents/Research/LLMxM2/.env


True

In [2]:
from pydantic import SecretStr

from langchain_openai import ChatOpenAI

from src.sage.runtime import SageRuntime
from src.sage.types import SageRuntimeConfig
from src.tools.catalog import AVAILABLE_TOOLS
from src.agent.controller import AgentController, ControllerConfig
from src.utils.console_logging import ConsoleLogger

/Users/snopoff/Documents/Research/LLMxM2/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/Users/snopoff/Documents/Research/LLMxM2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [4]:
logger = ConsoleLogger()

model = ChatOpenAI(
    model=f"gpt://{os.environ.get('YANDEX_FOLDER_ID')}/gpt-oss-20b/latest",
    base_url = "https://ai.api.cloud.yandex.net/v1",
    api_key = SecretStr(os.environ.get('YANDEX_API_KEY', ".env hasn't been loaded or YANDEX_API_KEY is missing")),
    temperature = 0,
    timeout = 60,
    max_retries = 2,
)

sage_runtime_config = SageRuntimeConfig()
sage_runtime = SageRuntime(config=sage_runtime_config, logger=logger)

with open('prompts/sage_skill/v2.md', 'r') as f:
    sage_skill = f.read()

with open('prompts/system_prompt/v3.md', 'r') as f:
    system_prompt = f.read()

tools = [tool(sage_runtime, sage_skill if key == "sage_exec" else "") for key, tool in AVAILABLE_TOOLS.items()]

controller_config = ControllerConfig(
    max_steps=4,
    progress_logs=True,
)

controller = AgentController(model=model, config=controller_config, tools=tools, system_prompt=system_prompt, logger=logger)

In [13]:
import json

benchmark_path = root / "data/processed/agent_benchmark_5.json"

with open(benchmark_path, 'r') as f:
    benchmark_problems = json.load(f)['problems']

benchmark_problems

[{'id': 'realmathbenchmark-00009',
  'index': 9,
  'arxiv_id': '2412.19095',
  'arxiv_version': 1,
  'link': 'http://arxiv.org/abs/2412.19095v1',
  'domain': 'graph theory',
  'task_type': 'chromatic polynomial of graph joins',
  'sage_affordance': 'Construct small graph pairs, compute chromatic polynomials of G1, G2, and their join, then fit the shifted-product formula.',
  'question': 'Let $G_1$ and $G_2$ be two graphs with orders $n_1$ and $n_2$, respectively, and let $G_1+G_2$ denote their join. What is the expression for the chromatic polynomial $\\mu(G_1+G_2;x)$ in terms of $x$, $n_1$, $n_2$, and the chromatic polynomials $\\mu(G_1,\\cdot)$ and $\\mu(G_2,\\cdot)$?',
  'answer': '$$\\mu(G_1+G_2;x)=\\frac{x(x-n_1-n_2)}{(x-n_1)(x-n_2)}\\,\\mu(G_1,x-n_2)\\,\\mu(G_2,x-n_1).$$'},
 {'id': 'realmathbenchmark-00352',
  'index': 352,
  'arxiv_id': '2412.19095',
  'arxiv_version': 1,
  'link': 'http://arxiv.org/abs/2412.19095v1',
  'domain': 'spectral graph theory',
  'task_type': 'distance

In [ ]:
prompt = """Construct a degree-19 polynomial p(x) in C[x] such that

    X := {p(x) = p(y)} ⊂ P¹ × P¹

has at least 3 irreducible components over C,
but not all components are linear.

Additional constraints:

• p(x) must be odd  
• p(x) must be monic  
• coefficients must be real  
• the linear coefficient must be −19

Then compute p(19)."""

In [ ]:
result = controller.solve(prompt)

In [ ]:
result.tool_traces[0]["arguments"]["code"]

In [ ]:
for trace in result.tool_traces:
    print(f"Turn: {trace['turn']}\nCode:{trace['arguments']['code']}")